# Notebook 06: Temporal Leakage Audit & Chronological Leave-Last-2-Out Splitting

## 1. Zero-Temporal-Leakage Invariant
Random train/test splits leak future reading behavior into training and artificially inflate NDCG. FedSocial V2 enforces strict per-user chronological Leave-Last-2-Out splitting in `ml/src/data/dataset.py`:
$$\forall u \in \mathcal{U}: \quad \max_{e \in \mathcal{E}_u^{\text{train}}} t(e) \le t(e_u^{\text{val}}) \le t(e_u^{\text{test}})$$

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Resolve repository root deterministically from ml/notebooks/
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DATA_DIR = os.path.join(REPO_ROOT, "ml", "data", "processed")
ARTIFACTS_DIR = os.path.join(REPO_ROOT, "ml", "artifacts")
VIZ_DIR = os.path.join(REPO_ROOT, "ml", "visualizations")
plt.rcParams["figure.dpi"] = 110
from ml.src.data.dataset import RecSysDataPipeline

pipeline = RecSysDataPipeline(
    articles_path=os.path.join(DATA_DIR, "articles.json"),
    interactions_path=os.path.join(DATA_DIR, "interactions.csv"),
    embeddings_path=os.path.join(DATA_DIR, "article_embeddings.npy"),
)

# Verify strict temporal monotonicity per user
violations = 0
for uid in pipeline.user_list:
    tr = pipeline.train_df[pipeline.train_df["user_id"] == uid]["event_timestamp"]
    va = pipeline.val_df[pipeline.val_df["user_id"] == uid]["event_timestamp"]
    te = pipeline.test_df[pipeline.test_df["user_id"] == uid]["event_timestamp"]
    if len(tr) and len(va) and tr.max() > va.min():
        violations += 1
    if len(va) and len(te) and va.max() > te.min():
        violations += 1

assert violations == 0, f"Temporal leakage detected in {violations} users!"
print(f"Train interactions      : {len(pipeline.train_df):,}")
print(f"Validation interactions : {len(pipeline.val_df):,}")
print(f"Test interactions       : {len(pipeline.test_df):,}")
print(f"Temporal Violations     : {violations} (STRICT ZERO LEAKAGE VERIFIED)")

Train interactions      : 4,665
Validation interactions : 100
Test interactions       : 100
Temporal Violations     : 0 (STRICT ZERO LEAKAGE VERIFIED)
